In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path(".")
DATA_MODEL = BASE_DIR / "data_model"

# Load asset returns
asset_returns = pd.read_csv(DATA_MODEL / "fact_asset_returns.csv")
asset_returns["price_date"] = pd.to_datetime(asset_returns["price_date"])

# Load portfolio definitions
scenarios = pd.read_csv(DATA_MODEL / "dim_portfolio_scenarios.csv")
allocs = pd.read_csv(DATA_MODEL / "fact_portfolio_allocations.csv")

# Merge returns with allocations on asset_symbol
merged = asset_returns.merge(
    allocs,
    on="asset_symbol",
    how="inner"
)

# weighted_return = daily_return * weight_pct
merged["weighted_return"] = merged["daily_return"] * merged["weight_pct"]

# Aggregate to scenario + date level
portfolio_daily = (
    merged
    .groupby(["scenario_id", "price_date"], as_index=False)
    .agg(portfolio_return=("weighted_return", "sum"))
)

# Optional: sanity check – print head
print(portfolio_daily.head())

# Save to CSV
portfolio_daily.to_csv(DATA_MODEL / "fact_portfolio_daily_returns.csv", index=False)
print("Saved:", DATA_MODEL / "fact_portfolio_daily_returns.csv")

   scenario_id price_date  portfolio_return
0            1 2021-01-01          0.000000
1            1 2021-01-04          0.009310
2            1 2021-01-05          0.005588
3            1 2021-01-06         -0.002680
4            1 2021-01-07          0.007599
Saved: data_model\fact_portfolio_daily_returns.csv
